# tokenbench on a T4: cross-checking servellm, then an open-loop rate sweep

GPU session required (Kaggle T4). ~10 min install, ~5 min load, ~20 min runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q "vllm==0.31.0"
%pip install -q "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
%pip install -q "git+https://github.com/NomanJafar/servellm"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")

## Launch vLLM (same config as servellm)

In [ ]:
from servellm.launch import ServerConfig, VllmServer
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096), log_path="results/startup.log")
server.start(ready_timeout_s=900)

## 1. Closed-loop sweep (should reproduce servellm's numbers)

In [ ]:
!tokenbench sweep --base-url http://127.0.0.1:8000 --model {MODEL} --out results/closed.json --levels 1,2,4,8,16,32,64 --num-requests 32 --prompt-tokens 256 --max-tokens 128 --slo-ttft-ms 500 --slo-tpot-ms 50 --plots

## 2. Open-loop rate sweep with the sample prompt distribution

Finds the arrival rate at which TTFT stops being bounded. 45 s per level.

In [ ]:
!tokenbench sweep --base-url http://127.0.0.1:8000 --model {MODEL} --out results/open.json --mode open --levels 0.5,1,2,4,8,16 --duration 45 --prompts sample --max-tokens 256 --slo-ttft-ms 1000 --slo-tpot-ms 50 --plots

In [ ]:
server.stop()

## 3. Look

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/closed.md").read()))
display(Markdown(open("results/open.md").read()))
for f in ["throughput_vs_load.png", "latency_vs_load.png", "goodput_vs_load.png"]:
    display(Image("results/" + f))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip